# B2-023 — Session 4: Denoising and Sampling

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

This session learns the reverse of Session 3's forward process.
It states the Gaussian posterior of one forward step, shows why matching it with a model of equal variance turns each training term into a squared error between means, and rewrites that error as the famous **ε-prediction** loss: predict the noise that was added.
It then builds the ancestral sampler that walks from pure noise back to data, with a worked reverse step, and closes with a fixed-seed tiny 2-D DDPM that trains and samples on CPU in about a second.
Practices after this session: p04, p10, p11, p16, p18, p19.

## 1. The reverse process

Generation runs the chain backwards. Start from $x_T\sim N(0,I)$, legitimate because $\bar\alpha_T\approx0$ (Session 3), and apply $T$ learned **reverse steps**
$$p_\theta(x_{t-1}\mid x_t)=N\big(\mu_\theta(x_t,t),\ \sigma_t^2I\big),\qquad t=T,T-1,\dots,1 .$$
Each step is Gaussian with a **learned mean** $\mu_\theta$ (a neural network of $x_t$ and $t$) and a **fixed variance** $\sigma_t^2$ chosen from the schedule; this unit never learns variances.

**Why a Gaussian reverse step is reasonable.** When $\beta_t$ is small, one forward step changes $x_{t-1}$ only a little, and the reverse of a small Gaussian step is itself close to Gaussian. This is the intuition behind using many small steps; the precise statement is beyond this course and the unit does not rely on it.

**What the network sees.** In this unit the network is an MLP on the concatenation of $x_t$ and the scaled timestep $t/T\in(0,1]$: input dimension $2+1=3$, architecture `3→64→64→2` with ReLU hidden layers (Book 1 `mlp-architecture`). Section 4 explains why its output is a noise prediction rather than $\mu_\theta$ directly.

**Checkpoint 1A.** Which parts of $p_\theta(x_{t-1}\mid x_t)$ are learned in this unit, and which are fixed?

**Checkpoint 1B.** For a batch of $B=256$ rows in 2-D, what is the shape of the network's input?

**Collected answers.** The mean $\mu_\theta(x_t,t)$ is learned (through the network); the variance $\sigma_t^2$ is fixed by the schedule. `(256,3)`: two coordinates of $x_t$ plus the scaled timestep $t/T$ as a third column.

## 2. The forward-process posterior (stated)

The forward process is fixed, so for a **known** clean row $x_0$ one can ask where $x_{t-1}$ was, given $x_t$. The answer is Gaussian. This unit **states** it without derivation (the derivation is Bayes' rule for Gaussians, which this course does not develop):
$$q(x_{t-1}\mid x_t,x_0)=N\big(\tilde\mu_t(x_t,x_0),\ \tilde\beta_tI\big),$$
$$\tilde\mu_t(x_t,x_0)=\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0+\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t,\qquad \tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t .$$

**Sanity checks.**

* $0\le\tilde\beta_t<\beta_t$: knowing $x_0$ makes the step back less uncertain than the step forward.
* **At $t=1$:** $\bar\alpha_0=1$ (the padding of Session 3), so $\tilde\beta_1=0$, the $x_t$ coefficient is $0$, and the $x_0$ coefficient is $\beta_1/(1-\bar\alpha_1)=\beta_1/\beta_1=1$. The posterior is a point mass at $x_0$: given $x_0$, there is nothing random left about it.
* The stored schedule gains its fourth entry, `beta_tilde`, with `beta_tilde[0] = beta_tilde[1] = 0`.

**Numbers** (Session 3's schedule, linear from $0.05$ to $0.35$, $T=4$): at $t=3$ the $x_0$ coefficient is $\frac{\sqrt{0.8075}\cdot0.25}{0.394375}=0.569642$, the $x_t$ coefficient is $\frac{\sqrt{0.75}\cdot0.1925}{0.394375}=0.422719$, and $\tilde\beta_3=\frac{0.1925}{0.394375}\cdot0.25=0.122029$, less than $\beta_3=0.25$.

**Checkpoint 2A.** Compute $\tilde\beta_2$ for the $T=4$ schedule ($\beta_2=0.15$, $\bar\alpha_1=0.95$, $\bar\alpha_2=0.8075$).

**Checkpoint 2B.** Why is $\tilde\beta_1=0$ in this unit's convention?

**Collected answers.** $\frac{0.05}{0.1925}\cdot0.15=0.038961$. Because $\bar\alpha_0=1$ makes the numerator $1-\bar\alpha_0$ vanish: the step from $x_1$ back to $x_0$ is deterministic once $x_0$ is known.

## 3. KL between Gaussians with equal variance

The diffusion training objective, like B2-022's VAE objective, comes from a variational bound on $\log p_\theta(x_0)$. Its terms for $t\ge2$ have the form
$$\mathrm{KL}\big(q(x_{t-1}\mid x_t,x_0)\ \big\|\ p_\theta(x_{t-1}\mid x_t)\big),$$
a KL between two Gaussians. B2-022 (Session 2, Section 6) derived the closed form for diagonal Gaussians, coordinate by coordinate:
$$\mathrm{KL}\big(N(\mu,\operatorname{diag}\sigma^2)\,\|\,N(\nu,\operatorname{diag}\tau^2)\big)=\sum_j\Big[\log\frac{\tau_j}{\sigma_j}+\frac{\sigma_j^2+(\mu_j-\nu_j)^2}{2\tau_j^2}-\frac12\Big].$$

**The equal-variance case.** If both covariances are $\sigma^2I$, the logarithm vanishes and the variance terms cancel the $\frac12$, leaving
$$\mathrm{KL}\big(N(\mu_1,\sigma^2I)\,\|\,N(\mu_2,\sigma^2I)\big)=\frac{\lVert\mu_1-\mu_2\rVert^2}{2\sigma^2}.$$
A KL between equal-variance Gaussians is a **scaled squared distance between means**. Showing this from the closed form, line by line, is part (a) of p16.

**Choosing $\sigma_t^2=\tilde\beta_t$.** If the model's reverse variance equals the posterior variance, every $t\ge2$ term is exactly this equal-variance case with $\sigma^2=\tilde\beta_t$, and training the model reduces to making $\mu_\theta(x_t,t)$ close to $\tilde\mu_t(x_t,x_0)$. This unit pins $\sigma_t^2=\tilde\beta_t$ for sampling as well (Section 6), so the derivation and the sampler agree.

**Checkpoint 3A.** Compute the KL between $N((1,0),0.5I)$ and $N((0,2),0.5I)$.

**Checkpoint 3B.** Why does the formula need the two variances to be equal?

**Collected answers.** $\lVert(1,-2)\rVert^2/(2\cdot0.5)=5/1=5$. With unequal variances the $\log(\tau/\sigma)$ and $\sigma^2/(2\tau^2)-\frac12$ terms no longer cancel, so the KL has extra terms that depend on the variances, not just a squared mean difference.

## 4. From mean matching to ε-prediction

**Rewrite the target mean with the noise.** By Session 3, a training input is $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$, so $x_0=\big(x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon\big)/\sqrt{\bar\alpha_t}$. Substituting this into the stated $\tilde\mu_t$ gives (stated here as part of the posterior; p16 asks you to use it)
$$\tilde\mu_t=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\Big).$$

**Parameterize the model the same way.** Let the network predict the noise, $\varepsilon_\theta(x_t,t)$, and define
$$\mu_\theta(x_t,t)=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big).$$
Then $\tilde\mu_t-\mu_\theta$ is a **schedule-only constant times $\varepsilon_\theta-\varepsilon$**, and Section 3 turns each $t\ge2$ term into
$$\lambda_t\,\big\lVert\varepsilon-\varepsilon_\theta(x_t,t)\big\rVert^2,$$
with a weight $\lambda_t$ that depends only on $\beta_t$, $\alpha_t$, $\bar\alpha_t$, and $\tilde\beta_t$. Deriving $\lambda_t$ is part (b) of p16. The network's **input** is $(x_t,\,t/T)$ and its **regression target** is the noise $\varepsilon$ that produced $x_t$, not $x_0$ and not $x_{t-1}$.

**Why $t=1$ is different.** At $t=1$, $\tilde\beta_1=0$, so the equal-variance KL would divide by zero: the argument does not apply. In the full bound, the $t=1$ term is a separate **decoder likelihood** $-\log p_\theta(x_0\mid x_1)$ with its own nonzero variance; this unit does not compute it. The simplified objective of Section 5 nevertheless uses the same ε squared error at $t=1$: there it is a **surrogate**, not a likelihood. Sampling's final step stays noiseless because $\tilde\beta_1=0$.

**The other common variance.** Many implementations sample with $\sigma_t^2=\beta_t$ instead. The $t\ge2$ KL then has unequal variances $\tilde\beta_t$ and $\beta_t$: it gains a term that does not depend on the model, and the mean-error coefficient changes from $\frac{1}{2\tilde\beta_t}$ to $\frac{1}{2\beta_t}$. The loss is still an ε squared error, with a different weight for each $t$. This unit pins $\tilde\beta_t$ everywhere.

**Checkpoint 4A.** What are the input and the regression target of the ε-prediction network?

**Checkpoint 4B.** Why does the equal-variance argument exclude $t=1$?

**Collected answers.** Input: the noisy row $x_t$ together with the scaled timestep $t/T$. Target: the standard-normal noise $\varepsilon$ used to make $x_t$ from $x_0$. Because $\tilde\beta_1=0$: the posterior is a point mass, the KL formula would divide by $\sigma^2=0$, and the bound's $t=1$ term is a different (decoder-likelihood) term.

## 5. The simplified training objective

The DDPM paper dropped the weights $\lambda_t$ and found that training improved. The **simplified objective** is
$$L_{\text{simple}}=E_{t\sim U\{1,\dots,T\},\ x_0\sim\text{data},\ \varepsilon\sim N(0,I)}\Big[\big\lVert\varepsilon-\varepsilon_\theta\big(\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\ t\big)\big\rVert^2\Big].$$

**One training update** (this unit's protocol, pinned in p10 and p18):

1. draw `t = torch.randint(1, T + 1, (B,), generator=noise_generator)`, one timestep per row, uniform on $1,\dots,T$;
2. draw `eps = torch.randn(B, 2, generator=noise_generator)`, in that order from the same dedicated generator;
3. `x_t = q_sample(x0, t, eps, alpha_bar)`, where the clean training rows enter only as `x0`;
4. predict `eps_hat = model(x_t, t_scaled)` with `t_scaled = (t.float() / T).unsqueeze(1)` of shape `(B,1)`;
5. the loss is `((eps_hat - eps) ** 2).mean()`, an **explicit mean over all $B\times d$ entries** (equivalently `F.mse_loss` with its default reduction, Book 1 `mse-loss`);
6. `zero_grad(set_to_none=True)` before the forward pass, then `backward()` and `step()` (Book 1 `autograd-training`).

**Reduction matters.** Summing instead of averaging multiplies the loss by $B\cdot d$ (and the effective learning rate of plain gradient descent with it); averaging only over rows multiplies it by $d$. Thresholds in p18 are stated for the mean over all entries.

**Worked exam-style example 1 (one loss value by hand).** Session 3's schedule, $x_0=(1,-1)$, $t=2$, $\varepsilon=(0.5,-0.5)$: $x_2=\sqrt{0.8075}(1,-1)+\sqrt{0.1925}(0.5,-0.5)=(1.117984,-1.117984)$. If the network outputs $\hat\varepsilon=(0.3,-0.2)$, the loss is $\frac12\big[(0.3-0.5)^2+(-0.2+0.5)^2\big]=\frac12(0.04+0.09)=0.065$. Using $x_0$ as the target instead would give $\frac12\big[(0.3-1)^2+(-0.2+1)^2\big]=0.565$: a different objective, which is the `ddpm_loss` defect p10's probes reject.

**Checkpoint 5A.** Recompute the worked example's loss if the model output were exactly $\varepsilon$.

**Checkpoint 5B.** A teammate uses `((eps_hat - eps) ** 2).sum()` on a batch of 256 two-dimensional rows. By what factor does their loss differ from this unit's?

**Collected answers.** $0$: a perfect noise prediction has zero squared error. By $256\cdot2=512$, since a sum over $B\times d$ entries equals $B d$ times their mean.

## 6. Ancestral sampling and a worked reverse step

**One reverse step**, `p_sample(model, x_t, t, z, schedule)` for an integer timestep $t$ shared by the whole batch:
$$\hat\varepsilon=\varepsilon_\theta(x_t,\,t/T),\qquad \text{mean}=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\varepsilon\Big),\qquad x_{t-1}=\begin{cases}\text{mean}+\sqrt{\tilde\beta_t}\,z, & t\ge2,\\ \text{mean}, & t=1.\end{cases}$$
The noise `z` is an explicit argument (a fresh standard-normal tensor of the batch's shape), and the standard deviation is $\sqrt{\tilde\beta_t}$ because $\sigma_t^2=\tilde\beta_t$. At $t=1$ no noise is added: $\tilde\beta_1=0$ anyway, and returning the mean exactly makes the final step deterministic.

**The loop**, `sample_loop(model, x_T, noises, schedule)`: start from `x = x_T`, then for $t=T,T-1,\dots,1$ (**descending**) set `x = p_sample(model, x, t, noises[t - 1], schedule)`, and return the final `x`, which is $x_0$. Here `noises` has shape `(T, B, d)` and `noises[t - 1]` is the noise used at timestep $t$; `noises[0]` belongs to $t=1$ and is ignored. Visiting timesteps in ascending order, or stopping at $t=2$, are the two classic loop bugs.

**Worked reverse step.** Session 3's schedule at $t=3$: $\beta_3=0.25$, $\alpha_3=0.75$, $\bar\alpha_3=0.605625$, $\tilde\beta_3=0.122029$. Take $x_3=(1.0,-0.5)$, a model output $\hat\varepsilon=(0.2,0.4)$, and $z=(1,-1)$.

1. Noise coefficient: $\beta_3/\sqrt{1-\bar\alpha_3}=0.25/0.627993=0.398094$.
2. Mean: $\big((1.0,-0.5)-0.398094\cdot(0.2,0.4)\big)/\sqrt{0.75}=(0.920381,-0.659238)/0.866025=(1.062765,-0.761222)$.
3. Add noise: $\sqrt{\tilde\beta_3}=0.349326$, so $x_2=(1.062765+0.349326,\ -0.761222-0.349326)=(1.412091,-1.110548)$.

At $t=1$ the same arithmetic would stop after step 2. The cell below repeats these numbers in code.

**Checkpoint 6A.** In the worked step, what would $x_2$ be if `z` were all zeros?

**Checkpoint 6B.** `noises` has shape `(50, 512, 2)`. Which slice is used at $t=50$, and which slice is never used?

**Collected answers.** The mean, $(1.062765,-0.761222)$. `noises[49]` at $t=50$; `noises[0]`, which belongs to the noiseless final step $t=1$, is never used.

In [ ]:
import math
import torch

T = 4
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(0.05, 0.35, T, dtype=torch.float64)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
beta_tilde[2:] = (1 - alpha_bar[1:-1]) / (1 - alpha_bar[2:]) * beta[2:]      # beta_tilde[1] = 0

t = 3
x_t = torch.tensor([1.0, -0.5], dtype=torch.float64)
eps_hat = torch.tensor([0.2, 0.4], dtype=torch.float64)
z = torch.tensor([1.0, -1.0], dtype=torch.float64)
coef = beta[t] / (1 - alpha_bar[t]).sqrt()
mean = (x_t - coef * eps_hat) / alpha[t].sqrt()
x_prev = mean + beta_tilde[t].sqrt() * z
print("beta_tilde:", [round(v, 6) for v in beta_tilde.tolist()])
print("coef", round(coef.item(), 6), " mean", [round(v, 6) for v in mean.tolist()], " x_{t-1}", [round(v, 6) for v in x_prev.tolist()])
assert torch.allclose(x_prev, torch.tensor([1.412091, -1.110548], dtype=torch.float64), atol=1e-6, rtol=0.0)

## 7. Worked laboratory: a fixed-seed tiny 2-D DDPM

The cell below puts Sessions 3–4 together at lesson scale on the unit's `mixture2d` training rows:

* the unit's schedule, $T=50$, linear from $0.001$ to $0.2$, stored with the $(T+1)$ convention and including `beta_tilde`;
* the `3→64→64→2` noise-prediction MLP on $(x_t,t/T)$, constructed right after `torch.manual_seed(SEED)`;
* **600** full-batch Adam updates with `lr=1e-3` and the update order of Section 5, drawing `t` then `eps` from a dedicated generator;
* 512 samples from fixed noise $x_T$ and `noises` drawn from a second generator, with the descending, noiseless-at-the-end loop of Section 6.

**What to look for.** The 50-step window means of the loss fall from about $0.6$ toward about $0.3$ and flatten: the loss does **not** go to zero, because predicting noise from a noisy input is genuinely uncertain at large $t$. The sample mean is near the origin, the sample covariance is near the data's (about $1$ on the diagonal, since the blobs sit at $(\pm1,\pm1)$), all four blobs receive samples, and about three quarters of the samples land within $0.6$ of a centre. The pure-noise starting points $x_T$, checked the same way, have only about a third inside a radius: the sampler, not the starting noise, creates the structure.

p18 and p19 run the full pinned protocol (1,500 updates, held-out denoising loss, statistical certificates); this cell is deliberately shorter and uses different noise seeds.

**Checkpoint 7A.** Why does the training loss level off well above zero?

**Checkpoint 7B.** The sampler is run under `torch.no_grad()`. What would go wrong without it?

**Collected answers.** At large $t$ the input is almost pure noise, so even a perfect model cannot recover $\varepsilon$ exactly from $x_t$; the minimum achievable mean squared error is positive. Autograd would record a graph through all 50 reverse steps, wasting memory and time for gradients that sampling never uses.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
import math
from torch import nn

T = 50
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(1e-3, 0.2, T, dtype=torch.float64)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
beta_tilde[2:] = (1 - alpha_bar[1:-1]) / (1 - alpha_bar[2:]) * beta[2:]
print(f"alpha_bar[T] = {alpha_bar[T].item():.6f}")

class TinyDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x_t, t_scaled):
        return self.net(torch.cat([x_t, t_scaled], dim=1))

x0 = generative_data.train_tensor("mixture2d")                       # (256, 2), clean training rows
torch.manual_seed(SEED)
model = TinyDenoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
noise_generator = torch.Generator().manual_seed(SEED + 20)
trace = []
for _ in range(600):
    t = torch.randint(1, T + 1, (x0.shape[0],), generator=noise_generator)
    eps = torch.randn(x0.shape[0], 2, generator=noise_generator)
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    optimizer.zero_grad(set_to_none=True)
    loss = ((model(x_t, (t.to(torch.float32) / T).unsqueeze(1)) - eps) ** 2).mean()
    loss.backward()
    optimizer.step()
    trace.append(loss.item())
print("loss window means every 100 steps:", [round(sum(trace[a:a + 50]) / 50, 3) for a in range(0, 600, 100)])

sample_generator = torch.Generator().manual_seed(SEED + 21)
x_T = torch.randn(512, 2, generator=sample_generator)
noises = torch.randn(T, 512, 2, generator=sample_generator)
with torch.no_grad():
    x = x_T
    for t in range(T, 0, -1):
        eps_hat = model(x, torch.full((x.shape[0], 1), t / T))
        mean = (x - beta[t].item() / math.sqrt(1 - alpha_bar[t].item()) * eps_hat) / math.sqrt(alpha[t].item())
        x = mean + math.sqrt(beta_tilde[t].item()) * noises[t - 1] if t >= 2 else mean

centers = generative_data.mixture_centers()
def inside_fraction(points):
    return (((points[:, None, :] - centers[None]) ** 2).sum(dim=2).min(dim=1).values <= 0.36).float().mean().item()
nearest = ((x[:, None, :] - centers[None]) ** 2).sum(dim=2).argmin(dim=1)
print("sample mean", [round(v, 3) for v in x.mean(dim=0).tolist()], " sample cov", [round(v, 3) for v in torch.cov(x.T).flatten().tolist()])
print("nearest-centre counts", torch.bincount(nearest, minlength=4).tolist())
print(f"fraction within 0.6 of a centre: samples {inside_fraction(x):.3f}   starting noise x_T {inside_fraction(x_T):.3f}")

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Regressing on $x_0$ (or $x_{t-1}$) instead of $\varepsilon$.* The sampler's mean formula assumes a noise prediction. Fix: the target is the `eps` that `q_sample` used.
2. *$\alpha_t$ and $\bar\alpha_t$ swapped in the mean.* The mean divides by $\sqrt{\alpha_t}$ and scales the noise by $\beta_t/\sqrt{1-\bar\alpha_t}$. Fix: write the formula from the stated posterior, and test at a $t$ where $\alpha_t\ne\bar\alpha_t$.
3. *Variance used as standard deviation.* `beta_tilde[t] * z` instead of `beta_tilde[t].sqrt() * z`. Fix: square root.
4. *Noise at the final step.* Adding $\sqrt{\beta_1}\,z$ at $t=1$ blurs every sample. Fix: return the mean at $t=1$.
5. *Loop direction and range.* Ascending timesteps, or `range(T, 1, -1)` that stops before $t=1$. Fix: `range(T, 0, -1)` with `noises[t - 1]`.
6. *Time-feature mismatch.* Training on $t/T$ but sampling with raw $t$ (or with $(t-1)/T$). Fix: one convention, $t/T$, everywhere.
7. *Held-out rows noised during training, or evaluation noise from the training stream.* Fix: training rows only enter `q_sample`'s `x0`; held-out losses use their own seeded `t` and `eps`.

**Exam connections.** p04 asks for the ε-network's input and target; p10 implements `ddpm_loss` with an explicit mean reduction; p11 implements `p_sample` and `sample_loop` with fixed noise and probes for the loop's order; p16 proves the equal-variance KL and derives the ε weight for $t\ge2$, explaining why $t=1$ is excluded; p18 and p19 train and sample the unit's DDPM with certificates; p23 repairs an implementation with an $\alpha$/$\bar\alpha$ confusion, a wrong $\sigma_t$, and an off-by-one loop.

**Checkpoint 8A.** Which loop bug leaves the final sample equal to $x_1$ instead of $x_0$?

**Checkpoint 8B.** Which practice derives the weight $\lambda_t$?

**Collected answers.** `for t in range(T, 1, -1)`, which stops after $t=2$ and never applies the final noiseless step. p16, part (b).

## 9. Forward-only deeper topic: faster samplers and learned variances

Forward-only deeper topic.
Ancestral sampling takes one network evaluation per timestep. Deterministic samplers such as DDIM reuse a trained ε-network but skip most timesteps, and later work learns the reverse variances instead of fixing them to $\tilde\beta_t$ or $\beta_t$; both ideas trade a little exactness for large speedups at image scale.
None of these samplers is a prerequisite, a permitted shortcut, or an assessed API in B2-023: every practice uses the pinned ancestral sampler with $\sigma_t^2=\tilde\beta_t$.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Which sampler and which variance do all B2-023 practices use?

**Checkpoint 9B.** Is DDIM assessed in this unit?

**Collected answers.** The ancestral sampler of Section 6 with $\sigma_t^2=\tilde\beta_t$. No; it is forward-only and not assessed.